In [ ]:
import numpy as np
import pandas as pd
import os
import joblib
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


DATA_PATH = "Dataset/Train.csv"
SAVE_DIR = "saved_gru_congestion_model"

N_PAST = 168

# Number of future hours predicted at once
N_FUTURE = 24

EPOCHS = 10
BATCH_SIZE = 64


os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
print("Loading dataset...")

df = pd.read_csv(DATA_PATH)

print("Dataset Shape:", df.shape)


df["TimeStamp"] = pd.to_datetime(
    df["TimeStamp"],
    format="%d/%m/%y %H:%M",
    errors="coerce"
)

df = df.dropna(
    subset=["TimeStamp"]
)

df = df.sort_values(
    "TimeStamp"
).reset_index(
    drop=True
)

Loading dataset...
Dataset Shape: (38373, 10)


In [ ]:
df["Hour"] = df["TimeStamp"].dt.hour
df["Day"] = df["TimeStamp"].dt.day
df["Month"] = df["TimeStamp"].dt.month
df["Year"] = df["TimeStamp"].dt.year
df["Weekday"] = df["TimeStamp"].dt.weekday


# Weekend

df["IsWeekend"] = (
    df["Weekday"] >= 5
).astype(int)


if "Holiday" in df.columns:

    df["IsHoliday"] = (
        df["Holiday"].notnull()
    ).astype(int)

else:

    df["IsHoliday"] = 0

In [ ]:
weather_encoder = LabelEncoder()

if "Weather" in df.columns:

    df["Weather"] = df["Weather"].fillna(
        "Unknown"
    )

    df["Weather_encoded"] = (
        weather_encoder.fit_transform(
            df["Weather"]
        )
    )

else:

    df["Weather_encoded"] = 0


weather_desc_encoder = LabelEncoder()

if "Weather_Desc" in df.columns:

    df["Weather_Desc"] = df[
        "Weather_Desc"
    ].fillna(
        "Unknown"
    )

    df["Weather_Desc_encoded"] = (
        weather_desc_encoder.fit_transform(
            df["Weather_Desc"]
        )
    )

else:

    df["Weather_Desc_encoded"] = 0

In [ ]:
df["Hour_sin"] = np.sin(
    2 * np.pi * df["Hour"] / 24
)

df["Hour_cos"] = np.cos(
    2 * np.pi * df["Hour"] / 24
)


df["Day_sin"] = np.sin(
    2 * np.pi * df["Day"] / 31
)

df["Day_cos"] = np.cos(
    2 * np.pi * df["Day"] / 31
)


df["Month_sin"] = np.sin(
    2 * np.pi * df["Month"] / 12
)

df["Month_cos"] = np.cos(
    2 * np.pi * df["Month"] / 12
)

In [ ]:
df["IsMorning"] = (
    (df["Hour"] >= 6) &
    (df["Hour"] < 12)
).astype(int)


df["IsAfternoon"] = (
    (df["Hour"] >= 12) &
    (df["Hour"] < 18)
).astype(int)


df["IsEvening"] = (
    (df["Hour"] >= 18) &
    (df["Hour"] < 24)
).astype(int)


columns_to_drop = []

for column in [
    "Date",
    "Holiday",
    "Weather",
    "Weather_Desc"
]:

    if column in df.columns:

        columns_to_drop.append(
            column
        )


df = df.drop(
    columns=columns_to_drop
)

In [ ]:

df = df.ffill().bfill()

TARGET = "Traffic_Vol"


feature_columns = [

    column

    for column in df.columns

    if column not in [
        "TimeStamp",
        TARGET
    ]

]


features = df[
    feature_columns
].copy()


target = df[
    TARGET
].copy()


print("\nNumber of Features:", len(feature_columns))


Number of Features: 22


In [ ]:
q33 = target.quantile(
    0.33
)

q66 = target.quantile(
    0.66
)


traffic_thresholds = {

    "low_threshold": q33,

    "moderate_threshold": q66

}


print("\nTraffic Thresholds")

print("Low <=", q33)

print("Moderate <=", q66)

print("High >", q66)


Traffic Thresholds
Low <= 2231.0
Moderate <= 4575.0
High > 4575.0


In [ ]:
feature_scaler = MinMaxScaler()

scaled_features = feature_scaler.fit_transform(
    features
)

target_scaler = MinMaxScaler()

scaled_target = target_scaler.fit_transform(
    target.values.reshape(
        -1,
        1
    )
)

In [ ]:
X = []

y = []


for i in range(

    N_PAST,

    len(df) - N_FUTURE + 1

):

    # Previous 168 hours

    X.append(

        scaled_features[
            i - N_PAST:i
        ]

    )


    # Next 24 hours

    y.append(

        scaled_target[
            i:i + N_FUTURE,
            0
        ]

    )


X = np.array(X)

y = np.array(y)


print("\nX Shape:", X.shape)

print("y Shape:", y.shape)


X Shape: (38182, 168, 22)
y Shape: (38182, 24)


In [ ]:
train_size = int(
    len(X) * 0.90
)


X_train = X[
    :train_size
]

X_test = X[
    train_size:
]


y_train = y[
    :train_size
]

y_test = y[
    train_size:
]

In [ ]:
model = tf.keras.Sequential([

    tf.keras.layers.Input(
        shape=(
            N_PAST,
            X.shape[2]
        )
    ),

    tf.keras.layers.GRU(
        128,
        return_sequences=True
    ),

    tf.keras.layers.Dropout(
        0.2
    ),

    tf.keras.layers.GRU(
        64,
        return_sequences=True
    ),

    tf.keras.layers.Dropout(
        0.2
    ),

    tf.keras.layers.GRU(
        32
    ),

    tf.keras.layers.Dense(
        64,
        activation="relu"
    ),

    tf.keras.layers.Dropout(
        0.2
    ),

    # Predict next 24 hours
    tf.keras.layers.Dense(
        N_FUTURE
    )

])


model.compile(

    optimizer="adam",

    loss="mse",

    metrics=[
        "mae"
    ]

)

model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 gru (GRU)                   (None, 168, 128)          58368     
                                                                 
 dropout (Dropout)           (None, 168, 128)          0         
                                                                 
 gru_1 (GRU)                 (None, 168, 64)           37248     
                                                                 
 dropout_1 (Dropout)         (None, 168, 64)           0         
                                                                 
 gru_2 (GRU)                 (None, 32)                9408      
                                                                 
 dense (Dense)               (None, 64)                2112      
                                                                 
 dropout_2 (Dropout)         (None, 64)               

In [ ]:
history = model.fit(

    X_train,

    y_train,

    epochs=EPOCHS,

    batch_size=BATCH_SIZE,

    validation_data=(
        X_test,
        y_test
    ),

    verbose=1

)

Epoch 1/10
537/537 [==============================] - 136s 254ms/step - loss: 0.0400 - mae: 0.1523 - val_loss: 0.0126 - val_mae: 0.0833
Epoch 2/10
537/537 [==============================] - 103s 191ms/step - loss: 0.0264 - mae: 0.1199 - val_loss: 0.0087 - val_mae: 0.0655
Epoch 3/10
537/537 [==============================] - 104s 193ms/step - loss: 0.0227 - mae: 0.1089 - val_loss: 0.0080 - val_mae: 0.0626
Epoch 4/10
537/537 [==============================] - 104s 194ms/step - loss: 0.0207 - mae: 0.1025 - val_loss: 0.0073 - val_mae: 0.0601
Epoch 5/10
537/537 [==============================] - 104s 194ms/step - loss: 0.0193 - mae: 0.0982 - val_loss: 0.0069 - val_mae: 0.0549
Epoch 6/10
537/537 [==============================] - 106s 197ms/step - loss: 0.0181 - mae: 0.0946 - val_loss: 0.0070 - val_mae: 0.0559
Epoch 7/10
537/537 [==============================] - 106s 198ms/step - loss: 0.0172 - mae: 0.0919 - val_loss: 0.0082 - val_mae: 0.0675
Epoch 8/10
537/537 [============================

In [ ]:
y_pred_scaled = model.predict(
    X_test
)


# Flatten before inverse transformation

y_pred = target_scaler.inverse_transform(

    y_pred_scaled.reshape(
        -1,
        1
    )

).reshape(

    y_pred_scaled.shape

)


y_actual = target_scaler.inverse_transform(

    y_test.reshape(
        -1,
        1
    )

).reshape(

    y_test.shape

)

120/120 [==============================] - 5s 30ms/step


In [ ]:
mse = mean_squared_error(

    y_actual.flatten(),

    y_pred.flatten()

)


mae = mean_absolute_error(

    y_actual.flatten(),

    y_pred.flatten()

)


r2 = r2_score(

    y_actual.flatten(),

    y_pred.flatten()

)


print("\n")
print("=" * 60)

print(
    "GRU MODEL PERFORMANCE"
)

print("=" * 60)


print("MSE:", mse)

print("MAE:", mae)

print("R2 Score:", r2)



GRU MODEL PERFORMANCE
MSE: 349163.0553746157
MAE: 385.82618977860403
R2 Score: 0.9142052757610609


In [ ]:
model.save(

    f"{SAVE_DIR}/gru_congestion_duration.keras"

)

joblib.dump(

    feature_scaler,

    f"{SAVE_DIR}/feature_scaler.pkl"

)


joblib.dump(

    target_scaler,

    f"{SAVE_DIR}/target_scaler.pkl"

)


joblib.dump(

    feature_columns,

    f"{SAVE_DIR}/feature_columns.pkl"

)


joblib.dump(

    N_PAST,

    f"{SAVE_DIR}/n_past.pkl"

)


joblib.dump(

    N_FUTURE,

    f"{SAVE_DIR}/n_future.pkl"

)


joblib.dump(

    traffic_thresholds,

    f"{SAVE_DIR}/traffic_thresholds.pkl"

)

joblib.dump(

    weather_encoder,

    f"{SAVE_DIR}/weather_encoder.pkl"

)


joblib.dump(

    weather_desc_encoder,

    f"{SAVE_DIR}/weather_desc_encoder.pkl"

)


df.to_pickle(

    f"{SAVE_DIR}/processed_history.pkl"

)

joblib.dump(

    history.history,

    f"{SAVE_DIR}/training_history.pkl"

)


print("\n")
print("=" * 60)

print(
    "GRU CONGESTION DURATION MODEL SAVED SUCCESSFULLY"
)

print("=" * 60)


print(
    "Saved in:",
    SAVE_DIR
)



GRU CONGESTION DURATION MODEL SAVED SUCCESSFULLY
Saved in: saved_gru_congestion_model
